# Value-at-Risk

## Import Libraries

In [ ]:
"""
Version 2 — keep original feel (Plotly), remove Alpha Vantage, add new VaR modes

What changed (minimal):
- Replaced Alpha Vantage with yfinance loader (no API key needed).
- Kept class names and overall structure: FinancialInstrument, Security, Portfolio.
- Fixed method bug: self.calc_total_return() call.
- Standardized TRADING_DAYS = 252 for annualization.
- Added VaR modes: historical h‑day, Gaussian, Student‑t (+ CVaR for Gaussian/historical).
- Added simple Kupiec backtest for 1‑day VaR.
- Kept Plotly visualizations (price, returns histogram with CVaR shading, portfolio sim plot).

Usage (example):
    python VAR_version2_plotly.py

Or import in a notebook and use the classes directly.
"""
from __future__ import annotations

# Core imports (kept simple & familiar)
import numpy as np
import pandas as pd
import yfinance as yf

# Plotly kept per your preference
import plotly.graph_objects as go

# Stats for VaR/backtest
from scipy.stats import norm, t as student_t, chi2

from dataclasses import dataclass
from typing import Iterable, Optional, Tuple, Dict

TRADING_DAYS = 252

# -----------------------------------
# Helper utilities (kept lightweight)
# -----------------------------------

def ann_return(daily_returns: pd.Series) -> float:
    if daily_returns is None or len(daily_returns) == 0:
        return float("nan")
    return (1 + daily_returns).prod() ** (TRADING_DAYS / len(daily_returns)) - 1


def ann_vol(daily_returns: pd.Series) -> float:
    if daily_returns is None or len(daily_returns) == 0:
        return float("nan")
    return float(daily_returns.std(ddof=1) * np.sqrt(TRADING_DAYS))


def max_drawdown_from_returns(daily_returns: pd.Series) -> float:
    if daily_returns is None or len(daily_returns) == 0:
        return float("nan")
    curve = (1 + daily_returns).cumprod()
    peak = curve.cummax()
    dd = curve / peak - 1
    return float(dd.min())


def load_prices_yf(ticker: str, start: str = "2015-01-01", end: Optional[str] = None) -> pd.DataFrame:
    """Adjusted prices via yfinance. Columns: Open, High, Low, Close, Volume, returns."""
    df = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    if df is None or df.empty:
        raise ValueError(f"No data returned for {ticker}")
    df = df.rename(columns=str.lower)
    df = df[~df.index.duplicated(keep="last")].sort_index()
    df["returns"] = df["close"].pct_change()
    return df.dropna()


def load_panel_yf(tickers: Iterable[str], start: str = "2015-01-01", end: Optional[str] = None) -> Tuple[pd.DataFrame, pd.DataFrame]:
    prices: Dict[str, pd.Series] = {}
    for t in tickers:
        df = load_prices_yf(t, start=start, end=end)
        prices[t] = df["close"].rename(t)
    prices_df = pd.DataFrame(prices).dropna(how="any").sort_index()
    rets_df = prices_df.pct_change().dropna()
    return prices_df, rets_df


# --------------------
# VaR implementations
# --------------------

def var_gaussian(mu: float, sigma: float, alpha: float = 0.95, horizon: int = 1) -> float:
    z = norm.ppf(alpha)
    mu_h = mu * horizon
    sig_h = sigma * np.sqrt(horizon)
    return mu_h - z * sig_h


def cvar_gaussian(mu: float, sigma: float, alpha: float = 0.95, horizon: int = 1) -> float:
    z = norm.ppf(alpha)
    mu_h = mu * horizon
    sig_h = sigma * np.sqrt(horizon)
    return mu_h - sig_h * norm.pdf(z) / (1 - alpha)


def var_student_t(mu: float, sigma: float, nu: int = 6, alpha: float = 0.95, horizon: int = 1) -> float:
    # Match variance of t to sigma^2, then scale for horizon
    zt = student_t.ppf(alpha, df=nu)
    sig_h = sigma * np.sqrt(horizon) * np.sqrt(nu / (nu - 2))
    mu_h = mu * horizon
    return mu_h - zt * sig_h


def var_cvar_historical(daily_returns: pd.Series, alpha: float = 0.95, horizon: int = 10) -> Tuple[float, float]:
    # True historical multi‑day compounding (no sqrt rule)
    hday = (1 + daily_returns).rolling(horizon).apply(lambda x: np.prod(x) - 1, raw=False).dropna()
    if hday.empty:
        return float("nan"), float("nan")
    var_ = float(np.quantile(hday, 1 - alpha))
    cvar_ = float(hday[hday <= var_].mean())
    return var_, cvar_


# ------------------
# VaR backtesting
# ------------------

def kupiec_pvalue(breaches: int, n: int, alpha: float = 0.95) -> float:
    if n <= 0:
        return float("nan")
    p = 1 - alpha
    x = breaches
    L0 = (1 - p) ** (n - x) * (p ** x)
    phat = x / n if n else 0.0
    if phat in (0.0, 1.0):
        # Edge cases → extreme LR
        return 0.0
    L1 = (1 - phat) ** (n - x) * (phat ** x)
    LRuc = -2 * (np.log(L0) - np.log(L1))
    return float(1 - chi2.cdf(LRuc, df=1))


def backtest_var_one_day(returns: pd.Series, var_series: pd.Series, alpha: float = 0.95) -> dict:
    df = pd.DataFrame({"ret": returns, "var": var_series}).dropna()
    breaches_mask = df["ret"] <= df["var"]
    b = int(breaches_mask.sum())
    n = int(len(df))
    return {
        "n": n,
        "breaches": b,
        "expected": (1 - alpha) * n,
        "breach_rate": b / n if n else float("nan"),
        "kupiec_p": kupiec_pvalue(b, n, alpha=alpha),
    }


# -----------------------------------
# Plot helpers (Plotly)
# -----------------------------------

def plot_price_series(df: pd.DataFrame, title: str = "Price") -> go.Figure:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df.index, y=df["close"], mode="lines", name="Close"))
    fig.update_layout(title=title, xaxis_title="Date", yaxis_title="Price")
    return fig


def plot_returns_hist(returns: pd.Series, alpha: float = 0.95, bins: int = 60, cvar_value: Optional[float] = None) -> go.Figure:
    fig = go.Figure()
    fig.add_trace(go.Histogram(x=returns.dropna(), nbinsx=bins, name="Daily returns"))
    # Shade tail if CVaR provided
    if cvar_value is not None:
        fig.add_vline(x=cvar_value, line_width=2, line_dash="dash", line_color="red")
    fig.update_layout(title=f"Returns histogram (alpha={alpha})", xaxis_title="Return", yaxis_title="Freq")
    return fig


def plot_mc_distribution(paths: np.ndarray, title: str = "MC horizon return dist") -> go.Figure:
    fig = go.Figure()
    fig.add_trace(go.Histogram(x=paths, nbinsx=60, name="Sim returns"))
    fig.update_layout(title=title, xaxis_title=f"Horizon return", yaxis_title="Count")
    return fig


# -----------------------------------
# Core classes (kept names)
# -----------------------------------

class FinancialInstrument:
    """Light wrapper you had for shared risk calcs/plots (kept minimal)."""

    @staticmethod
    def gaussian_var_cvar_from_series(returns: pd.Series, alpha: float = 0.95, horizon: int = 10) -> Tuple[float, float]:
        mu, sig = float(returns.mean()), float(returns.std(ddof=1))
        return var_gaussian(mu, sig, alpha=alpha, horizon=horizon), cvar_gaussian(mu, sig, alpha=alpha, horizon=horizon)

    @staticmethod
    def hist_var_cvar_from_series(returns: pd.Series, alpha: float = 0.95, horizon: int = 10) -> Tuple[float, float]:
        return var_cvar_historical(returns, alpha=alpha, horizon=horizon)


@dataclass
class Security(FinancialInstrument):
    ticker: str
    start: str = "2015-01-01"
    end: Optional[str] = None

    data: Optional[pd.DataFrame] = None
    returns: Optional[pd.Series] = None
    mu: Optional[float] = None
    sigma: Optional[float] = None
    total_return: Optional[float] = None

    def get_historical_price_data(self) -> pd.DataFrame:
        self.data = load_prices_yf(self.ticker, start=self.start, end=self.end)
        self.returns = self.data["returns"].copy()
        self.mu = float(self.returns.mean())
        self.sigma = float(self.returns.std(ddof=1))
        self.total_return = self.calc_total_return()  # fixed: no extra arg
        return self.data

    def calc_total_return(self) -> float:
        if self.returns is None or len(self.returns) == 0:
            return float("nan")
        return float((1 + self.returns).prod() - 1)

    # Plotly charts preserved
    def plot_price(self) -> go.Figure:
        if self.data is None:
            self.get_historical_price_data()
        return plot_price_series(self.data, title=f"{self.ticker} — Adjusted Close")

    def plot_returns_hist(self, alpha: float = 0.95, bins: int = 60) -> go.Figure:
        if self.returns is None:
            self.get_historical_price_data()
        # Optional: overlay CVaR (Gaussian)
        _, cvar_g = self.gaussian_var_cvar_from_series(self.returns, alpha=alpha, horizon=10)
        return plot_returns_hist(self.returns, alpha=alpha, bins=bins, cvar_value=cvar_g)


class Portfolio(FinancialInstrument):
    def __init__(self, tickers: Iterable[str], weights: Optional[Iterable[float]] = None, start: str = "2015-01-01", end: Optional[str] = None):
        self.tickers = list(tickers)
        self.start = start
        self.end = end
        self.prices, self.returns = load_panel_yf(self.tickers, start=start, end=end)
        if weights is None:
            self.weights = np.repeat(1 / len(self.tickers), len(self.tickers))
        else:
            w = np.asarray(list(weights), dtype=float)
            if len(w) != len(self.tickers):
                raise ValueError("weights length must match tickers")
            if np.isclose(w.sum(), 0):
                raise ValueError("weights sum to zero")
            self.weights = w / w.sum()

    @property
    def port_returns(self) -> pd.Series:
        return self.returns @ self.weights

    def summary(self) -> Dict[str, float]:
        r = self.port_returns
        return {
            "ann_return": ann_return(r),
            "ann_vol": ann_vol(r),
            "max_drawdown": max_drawdown_from_returns(r),
        }

    # ----- VaR interfaces (3 modes) -----
    def var(self, alpha: float = 0.95, horizon: int = 10, mode: str = "historical", nu: int = 6) -> Tuple[float, Optional[float]]:
        r = self.port_returns
        if mode == "historical":
            return var_cvar_historical(r, alpha=alpha, horizon=horizon)
        elif mode == "gaussian":
            mu, sig = float(r.mean()), float(r.std(ddof=1))
            return var_gaussian(mu, sig, alpha=alpha, horizon=horizon), cvar_gaussian(mu, sig, alpha=alpha, horizon=horizon)
        elif mode == "student_t":
            mu, sig = float(r.mean()), float(r.std(ddof=1))
            return var_student_t(mu, sig, nu=nu, alpha=alpha, horizon=horizon), None
        else:
            raise ValueError("mode must be one of {'historical','gaussian','student_t'}")

    # ----- Rolling 1‑day Gaussian VaR backtest -----
    def backtest_var_gaussian_1d(self, alpha: float = 0.95, lookback: int = 252) -> dict:
        r = self.port_returns
        roll_mu = r.rolling(lookback).mean()
        roll_sig = r.rolling(lookback).std(ddof=1)
        var_1d = roll_mu - norm.ppf(alpha) * roll_sig
        next_r = r.shift(-1)
        return backtest_var_one_day(next_r, var_1d, alpha=alpha)

    # ----- Simple Monte‑Carlo (Gaussian) for portfolio horizon return -----
    def mc_gaussian(self, n_paths: int = 20000, horizon_days: int = 10) -> np.ndarray:
        mu = self.returns.mean().values
        cov = self.returns.cov().values
        L = np.linalg.cholesky(cov)
        d = len(mu)
        agg = np.zeros(n_paths)
        # simulate daily compounding and aggregate to portfolio
        for _ in range(horizon_days):
            z = np.random.normal(size=(n_paths, d)) @ L.T
            daily = (mu + z)
            # update cumulative portfolio return
            step_port = (daily @ self.weights)
            agg = (1 + agg) * (1 + step_port) - 1
        return agg

    def plot_mc(self, n_paths: int = 40000, horizon_days: int = 10) -> go.Figure:
        sims = self.mc_gaussian(n_paths=n_paths, horizon_days=horizon_days)
        return plot_mc_distribution(sims, title=f"MC simulated portfolio returns — {horizon_days}d")


# ----------------------
# Small demo if run as script
# ----------------------
if __name__ == "__main__":
    # Example universe: feel free to change
    tickers = ["SPY", "TLT"]  # try TSX tickers like ["RY.TO","TD.TO","XEG.TO"] or tech ["AAPL","MSFT","NVDA"]
    pf = Portfolio(tickers, start="2018-01-01")
    print("Summary:", pf.summary())

    # VaR examples
    var_h, cvar_h = pf.var(alpha=0.95, horizon=10, mode="historical")
    var_g, cvar_g = pf.var(alpha=0.95, horizon=10, mode="gaussian")
    var_t, _ = pf.var(alpha=0.95, horizon=10, mode="student_t", nu=6)
    print(f"Historical 10d VaR/CVaR: {var_h:.4f} / {cvar_h:.4f}")
    print(f"Gaussian   10d VaR/CVaR: {var_g:.4f} / {cvar_g:.4f}")
    print(f"Student‑t  10d VaR:      {var_t:.4f}")

    # Backtest example
    bt = pf.backtest_var_gaussian_1d(alpha=0.95, lookback=252)
    print("Backtest (Kupiec p):", bt.get("kupiec_p"))

    # Plotly figures (open in browser/renderer)
    # Security example figure
    s = Security("SPY", start="2018-01-01")
    s.get_historical_price_data()
    fig_price = s.plot_price(); fig_price.show()
    fig_hist = s.plot_returns_hist(alpha=0.95); fig_hist.show()

    # Portfolio MC figure
    fig_mc = pf.plot_mc(n_paths=50000, horizon_days=10)
    fig_mc.show()


In [138]:
import numpy as np
import pandas as pd
import yfinance as yf


# Plotly kept per your preference
import plotly.graph_objects as go


# Stats for VaR/backtest
from scipy.stats import norm, t as student_t, chi2


from dataclasses import dataclass
from typing import Iterable, Optional, Tuple, Dict


TRADING_DAYS = 252

In [140]:
# get historical data 

def ann_return(daily_returns: pd.Series) -> float:
    if daily_returns is None or len(daily_returns) == 0:
        return float("nan")
    return (1 + daily_returns).prod() ** (TRADING_DAYS / len(daily_returns)) - 1


def ann_vol(daily_returns: pd.Series) -> float:
    if daily_returns is None or len(daily_returns) == 0:
        return float("nan")
    return float(daily_returns.std(ddof=1) * np.sqrt(TRADING_DAYS))


def max_drawdown_from_returns(daily_returns: pd.Series) -> float:
    if daily_returns is None or len(daily_returns) == 0:
        return float("nan")
    curve = (1 + daily_returns).cumprod()
    peak = curve.cummax()
    dd = curve / peak - 1
    return float(dd.min())


def load_prices_yf(ticker: str, start: str = "2015-01-01", end: Optional[str] = None) -> pd.DataFrame:
    """Adjusted prices via yfinance. Columns: Open, High, Low, Close, Volume, returns."""
    df = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    if df is None or df.empty:
        raise ValueError(f"No data returned for {ticker}")
    df = df.rename(columns=str.lower)
    df = df[~df.index.duplicated(keep="last")].sort_index()
    df["returns"] = df["close"].pct_change()
    return df.dropna()


def load_panel_yf(tickers: Iterable[str], start: str = "2015-01-01", end: Optional[str] = None) -> Tuple[pd.DataFrame, pd.DataFrame]:
    prices: Dict[str, pd.Series] = {}
    for t in tickers:
        df = load_prices_yf(t, start=start, end=end)
        prices[t] = df["close"].rename(t)

    prices_df = pd.DataFrame(prices).dropna(how="any").sort_index()
    rets_df = prices_df.pct_change().dropna()
    return prices_df, rets_df

In [142]:
# Implement VaR

def var_gaussian(mu: float, sigma: float, alpha: float = 0.95, horizon: int = 1) -> float:
    z = norm.ppf(alpha)
    mu_h = mu * horizon
    sig_h = sigma * np.sqrt(horizon)
    return mu_h - z * sig_h


def cvar_gaussian(mu: float, sigma: float, alpha: float = 0.95, horizon: int = 1) -> float:
    z = norm.ppf(alpha)
    mu_h = mu * horizon
    sig_h = sigma * np.sqrt(horizon)
    return mu_h - sig_h * norm.pdf(z) / (1 - alpha)


def var_student_t(mu: float, sigma: float, nu: int = 6, alpha: float = 0.95, horizon: int = 1) -> float:
    # Match variance of t to sigma^2, then scale for horizon
    zt = student_t.ppf(alpha, df=nu)
    sig_h = sigma * np.sqrt(horizon) * np.sqrt(nu / (nu - 2))
    mu_h = mu * horizon
    return mu_h - zt * sig_h


def var_cvar_historical(daily_returns: pd.Series, alpha: float = 0.95, horizon: int = 10) -> Tuple[float, float]:
    # True historical multi-day compounding (no sqrt rule)
    hday = (1 + daily_returns).rolling(horizon).apply(lambda x: np.prod(x) - 1, raw=False).dropna()
    if hday.empty:
        return float("nan"), float("nan")
    var_ = float(np.quantile(hday, 1 - alpha))
    cvar_ = float(hday[hday <= var_].mean())
    return var_, cvar_

In [144]:
#Var backtesting

def kupiec_pvalue(breaches: int, n: int, alpha: float = 0.95) -> float:
    if n <= 0:
        return float("nan")
    p = 1 - alpha
    x = breaches
    L0 = (1 - p) ** (n - x) * (p ** x)
    phat = x / n if n else 0.0
    if phat in (0.0, 1.0):
        # Edge cases → extreme LR
        return 0.0
    L1 = (1 - phat) ** (n - x) * (phat ** x)
    LRuc = -2 * (np.log(L0) - np.log(L1))
    return float(1 - chi2.cdf(LRuc, df=1))




def backtest_var_one_day(returns: pd.Series, var_series: pd.Series, alpha: float = 0.95) -> dict:
    df = pd.DataFrame({"ret": returns, "var": var_series}).dropna()
    breaches_mask = df["ret"] <= df["var"]
    b = int(breaches_mask.sum())
    n = int(len(df))
    return {
        "n": n,
        "breaches": b,
        "expected": (1 - alpha) * n,
        "breach_rate": b / n if n else float("nan"),
        "kupiec_p": kupiec_pvalue(b, n, alpha=alpha),
    }



In [146]:
# Plot helpers (Plotly)

def plot_price_series(df: pd.DataFrame, title: str = "Price") -> go.Figure:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df.index, y=df["close"], mode="lines", name="Close"))
    fig.update_layout(title=title, xaxis_title="Date", yaxis_title="Price")
    return fig




def plot_returns_hist(returns: pd.Series, alpha: float = 0.95, bins: int = 60, cvar_value: Optional[float] = None) -> go.Figure:
    fig = go.Figure()
    fig.add_trace(go.Histogram(x=returns.dropna(), nbinsx=bins, name="Daily returns"))
    # Shade tail if CVaR provided
    if cvar_value is not None:
        fig.add_vline(x=cvar_value, line_width=2, line_dash="dash", line_color="red")
    fig.update_layout(title=f"Returns histogram (alpha={alpha})", xaxis_title="Return", yaxis_title="Freq")
    return fig




def plot_mc_distribution(paths: np.ndarray, title: str = "MC horizon return dist") -> go.Figure:
    fig = go.Figure()
    fig.add_trace(go.Histogram(x=paths, nbinsx=60, name="Sim returns"))
    fig.update_layout(title=title, xaxis_title=f"Horizon return", yaxis_title="Count")
    return fig



In [154]:
#
class FinancialInstrument:
def port_returns(self) -> pd.Series:
return self.returns @ self.weights


def summary(self) -> Dict[str, float]:
r = self.port_returns
return {
"ann_return": ann_return(r),
"ann_vol": ann_vol(r),
"max_drawdown": max_drawdown_from_returns(r),
}


# ----- VaR interfaces (3 modes) -----
def var(self, alpha: float = 0.95, horizon: int = 10, mode: str = "historical", nu: int = 6) -> Tuple[float, Optional[float]]:
r = self.port_returns
if mode == "historical":
return var_cvar_historical(r, alpha=alpha, horizon=horizon)
elif mode == "gaussian":
mu, sig = float(r.mean()), float(r.std(ddof=1))
return var_gaussian(mu, sig, alpha=alpha, horizon=horizon), cvar_gaussian(mu, sig, alpha=alpha, horizon=horizon)
elif mode == "student_t":
mu, sig = float(r.mean()), float(r.std(ddof=1))
return var_student_t(mu, sig, nu=nu, alpha=alpha, horizon=horizon), None
else:
raise ValueError("mode must be one of {'historical','gaussian','student_t'}")


# ----- Rolling 1‑day Gaussian VaR backtest -----
def backtest_var_gaussian_1d(self, alpha: float = 0.95, lookback: int = 252) -> dict:
r = self.port_returns
roll_mu = r.rolling(lookback).mean()
roll_sig = r.rolling(lookback).std(ddof=1)
var_1d = roll_mu - norm.ppf(alpha) * roll_sig
next_r = r.shift(-1)
return backtest_var_one_day(next_r, var_1d, alpha=alpha)


# ----- Simple Monte‑Carlo (Gaussian) for portfolio horizon return -----
def mc_gaussian(self, n_paths: int = 20000, horizon_days: int = 10) -> np.ndarray:
mu = self.returns.mean().values
cov = self.returns.cov().values
L = np.linalg.cholesky(cov)
d = len(mu)
agg = np.zeros(n_paths)
# simulate daily compounding and aggregate to portfolio
for _ in range(horizon_days):
z = np.random.normal(size=(n_paths, d)) @ L.T
daily = (mu + z)
# update cumulative portfolio return
step_port = (daily @ self.weights)
agg = (1 + agg) * (1 + step_port) - 1
return agg


def plot_mc(self, n_paths: int = 40000, horizon_days: int = 10) -> go.Figure:
sims = self.mc_gaussian(n_paths=n_paths, horizon_days=horizon_days)
return plot_mc_distribution(sims, title=f"MC simulated portfolio returns — {horizon_days}d")



IndentationError: expected an indented block after class definition on line 2 (187815589.py, line 3)

In [ ]:
if __name__ == "__main__":
# Example universe: feel free to change
tickers = ["SPY", "TLT"] # try TSX tickers like ["RY.TO","TD.TO","XEG.TO"] or tech ["AAPL","MSFT","NVDA"]
pf = Portfolio(tickers, start="2018-01-01")
print("Summary:", pf.summary())


# VaR examples
var_h, cvar_h = pf.var(alpha=0.95, horizon=10, mode="historical")
var_g, cvar_g = pf.var(alpha=0.95, horizon=10, mode="gaussian")
var_t, _ = pf.var(alpha=0.95, horizon=10, mode="student_t", nu=6)
print(f"Historical 10d VaR/CVaR: {var_h:.4f} / {cvar_h:.4f}")
print(f"Gaussian 10d VaR/CVaR: {var_g:.4f} / {cvar_g:.4f}")
print(f"Student‑t 10d VaR: {var_t:.4f}")


# Backtest example
bt = pf.backtest_var_gaussian_1d(alpha=0.95, lookback=252)
print("Backtest (Kupiec p):", bt.get("kupiec_p"))


# Plotly figures (open in browser/renderer)
# Security example figure
s = Security("SPY", start="2018-01-01")
s.get_historical_price_data()
fig_price = s.plot_price(); fig_price.show()
fig_hist = s.plot_returns_hist(alpha=0.95); fig_hist.show()


# Portfolio MC figure
fig_mc = pf.plot_mc(n_paths=50000, horizon_days=10)
fig_mc.show()